In [151]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score , confusion_matrix)
from sklearn.ensemble import RandomForestClassifier 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

In [152]:
# load data set 
CSV_PATH  = "mail_17_dataset.csv"
df  = pd.read_csv(CSV_PATH)
print(df.head())

  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...


In [153]:
df = df.where(pd.notnull(df), "")

df["Category"] = df["Category"].str.lower().str.strip().map({"spam": 0, "ham": 1})

print(df.head())

   Category                                            Message
0         1  Go until jurong point, crazy.. Available only ...
1         1                      Ok lar... Joking wif u oni...
2         0  Free entry in 2 a wkly comp to win FA Cup fina...
3         1  U dun say so early hor... U c already then say...
4         1  Nah I don't think he goes to usf, he lives aro...


In [154]:
# split data

x = df["Message"].astype(str)
y = df["Category"].astype(int)

x_train , x_test  , y_train , y_test = train_test_split(
    x,y, test_size=0.2 , random_state=42
)

print("x_train : " , x_train.shape[0] )
print("x_test : " , x_test.shape[0] )





x_train :  4457
x_test :  1115


In [155]:
# text feature 
tfidf = TfidfVectorizer(min_df = 1 , stop_words = "english", lowercase=True)
x_train_features = tfidf.fit_transform(x_train)
x_test_features = tfidf.transform(x_test)


print("x_train_features : " , x_train_features.shape )
print("x_test_features : " , x_test_features.shape)




x_train_features :  (4457, 7440)
x_test_features :  (1115, 7440)


In [156]:
# train models

# logistic regression 
lr = LogisticRegression (max_iter=1000 , random_state=42)
lr.fit(x_train_features , y_train)
lr_pred = lr.predict(x_test_features)


# randomforestregressor

rf = RandomForestClassifier(n_estimators=200 ,random_state=42)
rf.fit(x_train_features , y_train)
rf_pred =  rf.predict(x_test_features)



# naive bayes
nb  = MultinomialNB()
nb.fit(x_train_features , y_train)
nb_pred  = nb.predict(x_test_features)

In [157]:
# Evaluate performance
def print_metrics (name , y_true , y_pred , pos_label= 0):
    acc = accuracy_score(y_true , y_pred)
    prec = precision_score (y_true , y_pred, pos_label=pos_label )
    rec = recall_score(y_true , y_pred , pos_label=pos_label)
    f1 =  f1_score(y_true , y_pred , pos_label=pos_label)
    
    print(f"{name }  performance : " )
    print(f"accurace : {acc:.3f}")
    print(f"precesion_score : {prec:.3f}  (positive = spam =0 )")
    print(f"recall_score: {rec:.3f}  (positive = spam =0 ) ")
    print(f"f1_score : {f1:.3f}   (positive = spam =0 )")

def print_confmat(name , y_true , y_pred ,) :
    cm = confusion_matrix(y_true , y_pred , labels=[1,0])
    cm_df=pd.DataFrame(
        cm,
        index =["Actual : Ham (1)" , "Actual Spam(0)"] ,
        columns = ["pred : ham (1)" , "pred_spam(0)"] 
   
    )
    print(f"\n{name} -confusion_matrix: \n{cm_df} ")

    

print_metrics("LogisticRegression : " , y_test , lr_pred )
print_confmat("LogisticRegression : " , y_test , lr_pred )


print_metrics("RandomForestclassifier : " , y_test , rf_pred )
print_confmat("RandomForestclassifier : " , y_test , rf_pred )


print_metrics("MultinomialNB : " , y_test , nb_pred  )
print_confmat("MultinomialNB : " , y_test , nb_pred )

   






LogisticRegression :   performance : 
accurace : 0.968
precesion_score : 1.000  (positive = spam =0 )
recall_score: 0.758  (positive = spam =0 ) 
f1_score : 0.863   (positive = spam =0 )

LogisticRegression :  -confusion_matrix: 
                  pred : ham (1)  pred_spam(0)
Actual : Ham (1)             966             0
Actual Spam(0)                36           113 
RandomForestclassifier :   performance : 
accurace : 0.983
precesion_score : 1.000  (positive = spam =0 )
recall_score: 0.872  (positive = spam =0 ) 
f1_score : 0.932   (positive = spam =0 )

RandomForestclassifier :  -confusion_matrix: 
                  pred : ham (1)  pred_spam(0)
Actual : Ham (1)             966             0
Actual Spam(0)                19           130 
MultinomialNB :   performance : 
accurace : 0.977
precesion_score : 1.000  (positive = spam =0 )
recall_score: 0.826  (positive = spam =0 ) 
f1_score : 0.904   (positive = spam =0 )

MultinomialNB :  -confusion_matrix: 
                  pred : ham

In [164]:
# sanity check

messege=  [
    
     "Free entry in 2 a weekly competition!" ,
     "I will meet you at the cafe tomorrow"  ,
     "Congratulations, you won a free ticket"  ,


]

x_test_sanity = tfidf.transform(messege)

print("Logistic Regression:", lr.predict(x_test_sanity))
print("Random Forest:", rf.predict(x_test_sanity))
print("Naive Bayes:", nb.predict(x_test_sanity))

Logistic Regression: [1 1 1]
Random Forest: [1 1 1]
Naive Bayes: [0 1 1]
